In [ ]:
import os
import sys
import django
PATH_PROJECT = fr"D:\PROGRAMMING\Projects\siakad\siakad_django"
sys.path.append(PATH_PROJECT)

os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'siakad_django.settings')
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"
django.setup()
from django.db import transaction

from django.db.models import Prefetch, Q

from django.conf import settings

In [ ]:
from academic.models import (
    KRS,
    DetailKRS, 
    PeriodeKRS,
    Mahasiswa,
    MataKuliah,
    Kelas,
    PenawaranKRS,
    StatusKRS
)

services

In [ ]:
from academic.krs.services import lihatDaftarKRS
from academic.krs.services.daftarHadir.daftarHadirService import (
    ExcelService, Query
)

In [ ]:
kelas_id = 130
template_path = settings.MEDIA_ROOT / "krs" / "template" / "DAFTAR_HADIR_TEMPLATE.xlsx"
dosen_nama = "Dodiman, S.Kom., M.Kom."
dosen_nidn = "0928079403"
hasil = ExcelService.excecute(
    kelas_id=kelas_id, template_path=template_path,
    DOSEN_PEMBINA=dosen_nama,
    NIDN=dosen_nidn
)
hasil.data

In [ ]:
with open("hasil.xlsx", "wb") as file:
    file.write(hasil.data.getvalue())

In [ ]:
svc = Query()
svc.mydata(kelas_id=130)['daftar_mahasiswa']
len(svc.mydata(kelas_id=130)['daftar_mahasiswa'])
svc.mydata(kelas_id=130)['mata_kuliah']
svc.mydata(kelas_id=130)['krs']
svc.mydata(kelas_id=130)['krs'].semester

In [ ]:
periode_krs = PeriodeKRS.objects.filter(status__iexact="aktif").first()
periode_krs

krs_id = 147
krs = KRS.objects.filter(id=krs_id).first()
krs

In [ ]:
svc = lihatDaftarKRS.Dataku()
res = svc.daftar_krs_by_kelas(periode_krs=periode_krs, kelas_id=130)
res
# len(res)
res[0]

penawaran_krs = PenawaranKRS.objects.filter(kelas__id=130).first()
kelas = penawaran_krs.kelas
periode_krs = penawaran_krs.periode_krs

kelas = Kelas.objects.get(id=130)
kelas.mata_kuliah
kelas.mata_kuliah.nama
kelas.mata_kuliah

periode_krs.id
periode_krs.tahun_ajaran
periode_krs.semester

openpyxl

In [ ]:
from pathlib import Path
import io

from openpyxl import Workbook, load_workbook
from openpyxl.worksheet.worksheet import Worksheet
from openpyxl.utils import column_index_from_string
from openpyxl.utils.cell import coordinate_to_tuple

In [ ]:
template_daftar_hadir = settings.MEDIA_ROOT / "krs" / "template" / "DAFTAR_HADIR_TEMPLATE.xlsx"
template_daftar_hadir.exists()

In [ ]:
# path_file = fr"D:\PROGRAMMING\Projects\siakad\siakad_django\academic\krs\services\daftarHadir"
# template_daftar_hadir = Path(path_file) / "DAFTAR_HADIR_TEMPLATE.xlsx"
# template_daftar_hadir.exists()

In [ ]:
wb = load_workbook(template_daftar_hadir, data_only=True)
ws_daftar_hadir = wb["KELAS"]
ws_jurnal_perkuliahan = wb["JURNAL PERKULIAHAN"]
print("")

In [ ]:
class ExcelService:
    def __init__(self):
        pass
    
    @staticmethod
    def simpan_excel(wb) -> io.BytesIO:
        output = io.BytesIO()
        wb.save(output)
        output.seek(0)
        return output
    
    @staticmethod
    def sisipkan_array(ws, *, kolom: str, data: list, baris_awal: int = 1):
        nomor_kolom = column_index_from_string(kolom)

        for row, value in enumerate(data, start=baris_awal):
            ws.cell(row=row, column=nomor_kolom, value=value)
            
    @staticmethod
    def sisipkan_nilai_by_coordinat(ws, *, coordinate: str, value:str):
        baris, kolom = coordinate_to_tuple(coordinate=coordinate)
        
        ws.cell(row=baris, column=kolom, value=value)

In [ ]:
fieldSet_daftar_hadir = {
    "MATA_KULIAH_SKS":  "E8",
    "SEMESTER_RUANG": "E9",
    "DOSEN_PEMBINA": "E10",
    "NIDN": "L10",
    "WAKTU": "E11",
    "NAMA_KELAS": "S8",
    "NIM": "B17",  # cell pertama
    "NAMA": "C17", # cell pertama
    "DOSEN_PEMBINA_NAMA": "O65",
    "DOSEN_PEMBINA_NIDN": "P66",
}

fieldSet_jurnal_perkuliahan = {
    "TAHUN_AKADEMIK": "E8",
    "MATA_KULIAH_SKS": "E9",
    "SEMESTER": "E10",
    "DOSEN_PEMBINA": "E11"
}

In [ ]:
output_bytesIO = ExcelService.simpan_excel(wb)
with open("hasil.xlsx", "wb") as file:
    file.write(output_bytesIO.getvalue())

query

In [ ]:
krs_qs = (
    KRS.objects
    .filter(
        semester=periode_krs.semester,
        tahun_ajaran=periode_krs.tahun_ajaran,
    )
    .select_related("mahasiswa")
    # .values_list("id", flat=True)
)
krs_qs

In [ ]:
krs_by_mahasiswa = {
    krs.mahasiswa.id: krs
    for krs in krs_qs
}
krs_by_mahasiswa
set(krs_by_mahasiswa)


In [ ]:
list_krs_ids = [65, 67, 95, 119, 161]
krs_temp = KRS.objects.filter(id__in=list_krs_ids).values_list("status")
krs_temp

In [ ]:
# semua mahasiswa aktif yang telah mengisi krs

mahasiswa_qs = (
        Mahasiswa.objects.filter(
        krs__semester=periode_krs.semester,
        krs__tahun_ajaran=periode_krs.tahun_ajaran,
        status__iexact="aktif",
    )
    .values_list("id", flat=True)
)
mahasiswa_qs
len(mahasiswa_qs)

In [ ]:
# tambahkan kelas
x = 24
krs_angkatan_x = (
    KRS.objects
    .filter(
        semester=periode_krs.semester,
        tahun_ajaran=periode_krs.tahun_ajaran,
        mahasiswa__nim__startswith=x
    )
    # .values_list("mahasiswa__nim", flat=True)
    
)
krs_angkatan_x
# len(krs_angkatan_x)


In [ ]:
# tambahkan kelas
x = 24
krs_angkatan_x = (
    KRS.objects
    .filter(
        semester=periode_krs.semester,
        tahun_ajaran=periode_krs.tahun_ajaran,
        mahasiswa__nim__startswith=x
    )
    .values_list("id", flat=True)
    
)
krs_angkatan_x
len(krs_angkatan_x)


In [ ]:
kelas_ids = [
    127, 144

]

In [ ]:
from itertools import product

krs_ids = krs_angkatan_x
kelas_ids = kelas_ids
krs_ids
list(product(krs_ids, kelas_ids))

In [ ]:
detail_krs_qs = (
    DetailKRS.objects
    .filter(krs__id__in=krs_angkatan_x)
    # .select_related("kelas", "kelas__mata_kuliah")
    # .delete()
)
detail_krs_qs
len(detail_krs_qs)

In [ ]:
detail_krs_qs

In [ ]:
mahasiswa_belum_krs = (
    Mahasiswa.objects
    .filter(
    #     krs__semester=periode_krs.semester,
    #     krs__tahun_ajaran=periode_krs.tahun_ajaran,
        status__iexact="aktif",
    )
    .exclude(id__in=mahasiswa_qs)
    .values_list("id", flat=True)
)
mahasiswa_belum_krs
len(mahasiswa_belum_krs)

In [ ]:
# buat krs otomatis
krs_list = [
    KRS(
        mahasiswa_id=mhs_id,
        semester=periode_krs.semester,
        tahun_ajaran=periode_krs.tahun_ajaran,
        status=StatusKRS.DISETUJUI
    )
    for mhs_id in mahasiswa_belum_krs
]
krs_list
# len(krs_list)

with transaction.atomic():
    KRS.objects.bulk_create(krs_list)
    print("berhasil")

In [ ]:
from academic.krs.services.lihatDaftarKRS import serializers


In [ ]:
query_set = (
    MataKuliah.objects.filter(
        kelas__penawarankrs__periode_krs=periode_krs,
        kelas__peserta__krs__semester=periode_krs.semester,
        kelas__peserta__krs__tahun_ajaran=periode_krs.tahun_ajaran,
    )
    .distinct()
    .prefetch_related(Prefetch("kelas_set", to_attr="daftar_kelas"))
)

query_set
serializers.MataKuliahKelasSerializer(instance=query_set, many=True).data
# serializers.DetailKRSKKelasSerializer(instance=query_set, many=True).data

services

In [ ]:
from academic.krs.services.lihatDaftarKRS import LihatDaftarKRS
from academic.krs.services.lihatDaftarKRS import serializers

In [ ]:
qs = LihatDaftarKRS().krs().data
qs
serializers.KRSSerializer(instance=qs, many=True).data

In [ ]:
svc = LihatDaftarKRS()
svc.lihat_daftar_krs()

query

In [ ]:
kelas_id=127
hasil_qs = (
    KRS.objects
    .filter(
        semester = periode_krs.semester,
        tahun_ajaran = periode_krs.tahun_ajaran,
        detail__kelas__id = kelas_id
    )
    .select_related("mahasiswa")
)
hasil_qs
sr = serializers.KRSMahasiswaPesertaSerializer(instance=hasil_qs, many=True)
sr.data

In [ ]:
detail_krs_qs = (
    DetailKRS.objects
    .filter(
        krs__semester=periode_krs.semester,
        krs__tahun_ajaran=periode_krs.tahun_ajaran
    )
)
detail_krs_qs